# [Notebook]Unihack26_VisA_Audit
Official VisA pixel audit on CPU, CC BY 4.0 data. Archive stays in Kaggle temp; exports hashes and split metadata only. No training, no model metrics, no automatic promotion or split reassignment.

In [ ]:
from pathlib import Path
import sys,json
SOURCES={'ml/__init__.py': '', 'ml/data/__init__.py': '', 'ml/data/visa_audit.py': '"""Audit official VisA on Kaggle; export metadata only, never the image archive."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport tarfile\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\nfrom urllib.request import urlopen\n\nimport numpy as np\nfrom PIL import Image\n\nREVISION = "2a692ab575001cbde74d402d897a7286086c6199"\nROOT_URL = f"https://raw.githubusercontent.com/amazon-science/spot-diff/{REVISION}/"\nSPLIT_SHA = "a48557e6033318cb90556f706196bc9d247a776a23ea51aecee5a80dd0332995"\nREADME_SHA = "803d1a86c80fcdebc63dfabd041f78336c433ca635b735668aadb24bfc124917"\nARCHIVE_URL = "https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar"\nARCHIVE_BYTES = 1_929_840_640\n\n\ndef fetch_metadata(relative, expected):\n    with urlopen(ROOT_URL + relative, timeout=60) as response:\n        content = response.read(2_000_001)\n    if len(content) > 2_000_000 or hashlib.sha256(content).hexdigest() != expected:\n        raise ValueError("OFFICIAL_METADATA_CHECKSUM_MISMATCH")\n    return content\n\n\ndef normalized_member(name):\n    while name.startswith("./"):\n        name = name[2:]\n    path = PurePosixPath(name)\n    if path.is_absolute() or ".." in path.parts or "\\\\" in name or ":" in name:\n        raise ValueError("UNSAFE_ARCHIVE_MEMBER")\n    if path.parts and path.parts[0] == "VisA":\n        path = PurePosixPath(*path.parts[1:])\n    return path.as_posix()\n\n\ndef audit_archive(archive, split_bytes, output, expected_count=10821):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    split_rows = list(csv.DictReader(io.StringIO(split_bytes.decode("utf-8-sig"))))\n    expected = {row["image"]: row for row in split_rows}\n    if len(split_rows) != len(expected) or len(expected) != expected_count:\n        raise ValueError("UNEXPECTED_OFFICIAL_SPLIT")\n    if any(row["split"] not in {"train", "test"} or (row["split"] == "train" and row["label"] != "normal")\n           for row in split_rows):\n        raise ValueError("NONSTANDARD_TRAIN_TEST_SPLIT")\n    records, seen = [], set()\n    hashes = {"sha256": defaultdict(list), "rgb_sha256": defaultdict(list)}\n    with tarfile.open(archive, "r:*") as tar:\n        for member in tar:\n            name = normalized_member(member.name)\n            if name not in expected:\n                continue\n            if name in seen or not member.isfile() or not 0 < member.size <= 32_000_000:\n                raise ValueError("INVALID_OR_DUPLICATE_IMAGE_MEMBER")\n            seen.add(name)\n            with tar.extractfile(member) as stream:\n                raw = stream.read(32_000_001)\n            if len(raw) != member.size:\n                raise ValueError("IMAGE_SIZE_MISMATCH")\n            with Image.open(io.BytesIO(raw)) as image:\n                rgb = image.convert("RGB")\n                rgb.load()\n                pixels = np.asarray(rgb)\n                thumbnail = np.asarray(rgb.convert("L").resize((9, 8)))\n                bits = (thumbnail[:, 1:] > thumbnail[:, :-1]).flatten()\n                dhash = sum(int(value) << index for index, value in enumerate(bits))\n                shape = list(pixels.shape)\n                decoded_hash = hashlib.sha256(pixels.tobytes()).hexdigest()\n            row = expected[name]\n            record = {"image_path": name, "category": row["object"], "source_split": row["split"],\n                      "role": row["split"], "label": row["label"], "shape": shape,\n                      "sha256": hashlib.sha256(raw).hexdigest(), "rgb_sha256": decoded_hash,\n                      "dhash64": f"{dhash:016x}", "bytes": len(raw),\n                      "physical_instance_id": None}\n            records.append(record)\n            for kind in hashes:\n                hashes[kind][record[kind]].append(record)\n            if len(records) % 1000 == 0:\n                print(json.dumps({"audited_images": len(records), "expected": expected_count}), flush=True)\n    if seen != set(expected):\n        raise ValueError("OFFICIAL_IMAGES_MISSING_FROM_ARCHIVE")\n    duplicate_groups, cross_split = [], []\n    for kind, groups in hashes.items():\n        for digest, items in groups.items():\n            if len(items) > 1:\n                item = {"kind": kind, "hash": digest, "images": [r["image_path"] for r in items],\n                        "roles": sorted({r["role"] for r in items})}\n                duplicate_groups.append(item)\n                if len(item["roles"]) > 1:\n                    cross_split.append(item)\n    manifest = output / "images.jsonl"\n    manifest.write_text("".join(json.dumps(row, sort_keys=True) + "\\n" for row in sorted(records, key=lambda r: r["image_path"])),\n                        encoding="utf-8")\n    (output / "duplicates.json").write_text(json.dumps(duplicate_groups, indent=2), encoding="utf-8")\n    report = {"status": "PIXEL_AUDIT_COMPLETE", "images": len(records),\n              "counts": dict(Counter(r["role"] for r in records)), "categories": dict(Counter(r["category"] for r in records)),\n              "manifest_sha256": hashlib.sha256(manifest.read_bytes()).hexdigest(),\n              "duplicate_groups_by_hash_type": dict(Counter(r["kind"] for r in duplicate_groups)),\n              "cross_split_duplicate_groups": len(cross_split),\n              "split_sha256": hashlib.sha256(split_bytes).hexdigest(), "source_revision": REVISION,\n              "license": "CC BY 4.0 (data, per pinned official README)",\n              "training_ready": False, "test_used_for_training_or_metrics": False,\n              "limits": ["Near-duplicate and cross-dataset review pending.",\n                         "Official split has no verified physical-instance lineage IDs.",\n                         "Industrial anomaly data is not paired parcel handover ground truth.",\n                         "Mask files are not audited or used here; only normal official train is a pretraining candidate."]}\n    (output / "audit.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    return report\n\n\ndef run(output="/kaggle/working/visa-audit", temp="/kaggle/temp/visa-audit"):\n    output, temp = Path(output), Path(temp)\n    output.mkdir(parents=True, exist_ok=True)\n    temp.mkdir(parents=True, exist_ok=True)\n    split = fetch_metadata("split_csv/1cls.csv", SPLIT_SHA)\n    readme = fetch_metadata("README.md", README_SHA)\n    if b"CC BY 4.0" not in readme:\n        raise ValueError("DATA_LICENSE_NOT_FOUND")\n    (output / "1cls.csv").write_bytes(split)\n    (output / "official-README.md").write_bytes(readme)\n    archive = temp / "VisA_20220922.tar"\n    digest, total = hashlib.sha256(), 0\n    with urlopen(ARCHIVE_URL, timeout=60) as response:\n        with archive.open("wb") as stream:\n            for chunk in iter(lambda: response.read(1024 * 1024), b""):\n                total += len(chunk)\n                if total > ARCHIVE_BYTES:\n                    raise ValueError("OFFICIAL_ARCHIVE_SIZE_CHANGED")\n                stream.write(chunk)\n                digest.update(chunk)\n    if total != ARCHIVE_BYTES:\n        raise ValueError("OFFICIAL_ARCHIVE_SIZE_CHANGED")\n    download = {"url": ARCHIVE_URL, "bytes": total, "sha256": digest.hexdigest(),\n                "archive_location": "KAGGLE_TEMP_ONLY_NOT_EXPORTED"}\n    (output / "download.json").write_text(json.dumps(download, indent=2), encoding="utf-8")\n    report = audit_archive(archive, split, output)\n    print(json.dumps(report), flush=True)\n    return report\n'}
root=Path('/kaggle/working/_src')
for name,source in SOURCES.items():
 p=root/name
 p.parent.mkdir(parents=True,exist_ok=True)
 p.write_text(source,encoding='utf-8')
sys.path.insert(0,str(root))
Path('/kaggle/working/source-hashes.json').write_text(json.dumps({'ml/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/visa_audit.py': '7e021357a360737ad93b444b0baa5cbd119139c6de4e5d04843deae5f1180ed5'},indent=2))


In [ ]:
from ml.data.visa_audit import run
run()